# Tichy's Einblick
Load Tichy's Einblick CSV files and create a DataFrame named `df`.

In [ ]:
from pathlib import Path
import pandas as pd

def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent):
        if (candidate / "data" / "raw" / "Alternative Medien").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw/Alternative Medien")

def read_csv_resilient(csv_path: Path) -> pd.DataFrame:
    for encoding in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(
                csv_path,
                encoding=encoding,
                low_memory=False,
                on_bad_lines="skip",
            )
        except Exception:
            pass

    for encoding in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            return pd.read_csv(
                csv_path,
                encoding=encoding,
                sep=None,
                engine="python",
                low_memory=False,
                on_bad_lines="skip",
            )
        except Exception:
            pass

    raise ValueError(f"Could not read: {csv_path}")

PROJECT_ROOT = resolve_project_root()
BASE_DIR = PROJECT_ROOT / "data" / "raw" / "Alternative Medien"

SOURCE_NAME = "Tichy's Einblick"
SOURCE_DIR = BASE_DIR / SOURCE_NAME
csv_files = sorted(SOURCE_DIR.rglob("*.csv"))

parts = []
for csv_file in csv_files:
    part = read_csv_resilient(csv_file)
    part["source"] = SOURCE_NAME
    part["source_file"] = csv_file.name
    parts.append(part)

df = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()

print(f"Loaded {len(df)} rows from {len(csv_files)} file(s) in {SOURCE_DIR}")
df.head()


## BERTopic

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if project_root.name == "data preprocessing":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))


In [ ]:
from BERTopic.bertopic_config import BERTopicConfig
from BERTopic.bertopic_pipeline import run_bertopic_pipeline


In [ ]:
# Prepare Tichy's Einblick dataframe for topic modeling
df_clean = df[["title", "date", "url", "article_text"]].copy()
df_clean["day"] = pd.to_datetime(df_clean["date"], errors="coerce")
df_clean["article_text"] = (
    df_clean["article_text"]
    .astype("string")
    .str.replace("\n", " ", regex=False)
    .str.strip()
)

df_clean = df_clean.dropna(subset=["day", "url", "article_text"]).copy()
df_clean = df_clean[df_clean["article_text"] != ""].copy()

df_clean = df_clean[
    (df_clean["day"] >= "2025-08-01") & (df_clean["day"] <= "2026-01-31")
].copy()

print(f"Rows prepared for BERTopic: {len(df_clean):,}")
df_clean[["title", "day", "url", "article_text"]].head()


In [ ]:
result = run_bertopic_pipeline(
    df=df_clean,
    text_col="article_text",
    config=BERTopicConfig(),
    id_col="url",
    source_name="Tichys_Einblick",
)


In [ ]:
topic_info = result["topic_info"]
topic_info.head(20)


In [ ]:
doc_info = result["doc_info"]
doc_info.head()


In [ ]:
tichys_topics = df_clean.reset_index().merge(
    doc_info[["original_index", "Topic", "Name", "Probability"]],
    left_on="index",
    right_on="original_index",
    how="left",
)

tichys_topics.head()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(14, 10))

# Exclude topic -1 (outliers) for cleaner visualization
topic_counts = topic_info[topic_info["Topic"] != -1].copy()
topic_counts = topic_counts.nlargest(20, "Count").sort_values("Count", ascending=True)

ax.barh(topic_counts["Name"], topic_counts["Count"], color="#1f77b4", edgecolor="white")

ax.set_xlabel("Number of Documents", fontsize=12)
ax.set_ylabel("Topic", fontsize=12)
ax.set_title("Tichy's Einblick BERTopic: Document Count per Topic", fontsize=14)
ax.grid(axis="x", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
# Prepare data for topics over time
prepared_dates = pd.to_datetime(result["prepared_documents"]["day"], errors="coerce")
valid_time_mask = prepared_dates.notna().tolist()
docs_for_time = [doc for doc, ok in zip(result["docs"], valid_time_mask) if ok]
timestamps_for_time = [ts for ts in prepared_dates.tolist() if pd.notna(ts)]

topics_over_time = result["topic_model"].topics_over_time(
    docs_for_time,
    timestamps_for_time,
    nr_bins=6,
)

# Get top 10 topics by count (excluding -1)
top_10_topics = topic_info[topic_info["Topic"] != -1].nlargest(10, "Count")["Topic"].tolist()

# Filter for top 10 topics
topics_over_time_filtered = topics_over_time[topics_over_time["Topic"].isin(top_10_topics)]

# Create topic name mapping
topic_name_map = dict(zip(topic_info["Topic"], topic_info["Name"]))

# Plot
fig, ax = plt.subplots(figsize=(14, 8))
sns.set_theme(style="whitegrid", context="talk")

palette = sns.color_palette("husl", n_colors=10)

for i, topic in enumerate(top_10_topics):
    topic_data = topics_over_time_filtered[topics_over_time_filtered["Topic"] == topic]
    label = topic_name_map.get(topic, f"Topic {topic}")
    # Shorten label for legend
    short_label = label.split("_")[1] if "_" in label else label
    ax.plot(
        topic_data["Timestamp"],
        topic_data["Frequency"],
        marker="o",
        linewidth=2.5,
        markersize=8,
        color=palette[i],
        label=short_label,
    )

ax.set_xlabel("Time", fontsize=14)
ax.set_ylabel("Number of Documents", fontsize=14)
ax.set_title("Tichy's Einblick: Top 10 Topics Over Time (Aug 2025 - Jan 2026)", fontsize=16, fontweight="bold")
ax.legend(title="Topic", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=10)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
